# Historical analysis — public source copy
Outputs and execution counts were cleared for publication privacy. Original evidence remains local.
Do not Run All modeling notebooks, retrain frozen models or reevaluate the consumed test.
Dataset/model assets are withheld; see the release README.

# 02 — Data cleaning, exploratory analysis and business insights

**Phase 2 only.** This notebook reuses the existing package and IBM source CSV. It produces a validated analytical dataset, descriptive KPIs, univariate/bivariate/multivariate summaries, and saved Matplotlib/Seaborn charts. It does not fit preprocessing, train models, make predictions, or implement SQL or Streamlit.

Run all cells with the project `.venv`. The CSV must already exist; use the Phase 1 downloader if needed. All quantitative observations below are computed from the data rather than typed-in evaluation results.

In [ ]:
from pathlib import Path
import sys
import hashlib
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path.cwd().resolve()
if not (ROOT / "src/churn_analytics").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/churn_analytics").is_dir(), "Launch from project root or notebooks/"
sys.path.insert(0, str(ROOT / "src"))
from churn_analytics.data import DEFAULT_RAW_PATH, load_dataset, validate_dataset
from churn_analytics.cleaning import (clean_dataset, build_quality_report,
    load_cleaned_dataset, validate_cleaned_dataset)
from churn_analytics.analytics import (customer_kpis, churn_by_segment, categorical_summary,
    numerical_summary, association_tests, correlation_tables)
from churn_analytics.phase2 import run_phase2

REPORTS = ROOT / "reports"
raw_sha = hashlib.sha256(DEFAULT_RAW_PATH.read_bytes()).hexdigest()
print("Python executable:", sys.executable)
raw = load_dataset()
print("Original dimensions:", raw.shape)
display(raw.head())

## A. Dataset overview and feature meanings

The unit of analysis is one customer record. Feature names describe recorded attributes, not experimentally assigned treatments. The source provides no calendar event series for this analysis.

| Feature(s) | Analytical meaning |
| --- | --- |
| customerID | Record identifier; excluded from statistical correlations and future predictive features |
| gender, SeniorCitizen, Partner, Dependents | Recorded demographic/household attributes; SeniorCitizen is binary (0/1) |
| tenure | Recorded tenure in months |
| PhoneService, MultipleLines | Phone subscription and multiple-line status |
| InternetService | DSL, Fiber optic, or No internet service |
| OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies | Add-on status; No internet service is a meaningful structural category |
| Contract | Recorded contract type |
| PaperlessBilling, PaymentMethod | Billing and payment attributes |
| MonthlyCharges, TotalCharges | Recorded monthly and cumulative charge measures, in dataset charge units |
| Churn | Yes/No outcome label; no estimated probability |

`tenure`, `MonthlyCharges` and `TotalCharges` are continuous/count measures for numerical EDA. The other non-ID predictors, including SeniorCitizen, are categorical. `ChurnFlag` added later is the outcome in numeric form, never a predictive feature.

In [ ]:
phase1 = validate_dataset(raw)
assert not phase1["errors"], phase1["errors"]
print("Duplicate customer IDs:", phase1["duplicate_customer_ids"])
print("Exact duplicate records:", int(raw.duplicated().sum()))
display(pd.DataFrame({"dtype": raw.dtypes.astype(str),
                      "nulls": raw.isna().sum(),
                      "whitespace_only": pd.Series(phase1["blank_values"])}))
display(raw.describe(include="all").T)

## Investigate blank TotalCharges before choosing a policy

A whitespace-only value is not the same as a pandas null. Inspect all affected records and their tenure/outcome. Possible explanations include billing-cycle timing for newly enrolled customers or an omitted source value. The snapshot cannot determine which explanation is correct or prove that the missing charge equals zero.

In [ ]:
blank = raw.TotalCharges.astype("string").str.fullmatch(r"\s*").fillna(False)
display(raw.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])
display(pd.Series(phase1["total_charges_investigation"]))
display(Markdown(f"**Observed:** {int(blank.sum())} blank charges; "
                 f"{int((blank & raw.tenure.eq(0)).sum())} occur at zero tenure. "
                 f"Their target counts are {raw.loc[blank, 'Churn'].value_counts().to_dict()}."))

## Cleaning decisions and before/after quality

**Chosen policy:** preserve every customer, convert blank TotalCharges to numeric missing values, and flag missingness. Do not impute zero or exclude rows. Charge summaries explicitly use observed-value denominators.

The deterministic cleaner strips outer whitespace, validates IDs, domains, numeric ranges and phone/internet consistency, and rejects invalid records for investigation. It does not silently discard malformed values. Valid extreme observations remain. Four analytical columns are added: ChurnFlag, TotalChargesMissing, TenureGroup and MonthlyChargeGroup. Fixed groups are documented in the guide; no quantiles or other parameters are learned.

Missing means unavailable information. Invalid means a value contradicts the schema/domain, such as a negative tenure or `Churn="Maybe"`. They require different handling.

In [ ]:
cleaned = clean_dataset(raw)
quality = build_quality_report(raw, cleaned)
assert not validate_cleaned_dataset(cleaned)["errors"]
pd.testing.assert_frame_equal(cleaned, clean_dataset(cleaned))
print("Before:", raw.shape, "After:", cleaned.shape)
display(pd.DataFrame({"before_dtype": raw.dtypes.astype(str),
                      "after_dtype": cleaned.dtypes.astype(str),
                      "before_nulls": raw.isna().sum(),
                      "after_nulls": cleaned.isna().sum()}))
display(pd.DataFrame(quality["alternatives"]))
print("Alternative churn_rate is a proportion, not a percentage.")
print("Cleaned validation:", quality["cleaned"])

## Reproduce the saved dataset and report artifacts

This command refreshes `data/processed/telco_churn_cleaned.csv`, the quality report, KPI JSON, tables, 17 charts and the business report. It verifies CSV reload and the raw fingerprint. Re-running is safe. The original Phase 1 module and notebook remain in place.

In [ ]:
manifest = run_phase2()
cleaned = load_cleaned_dataset()
assert hashlib.sha256(DEFAULT_RAW_PATH.read_bytes()).hexdigest() == raw_sha
print("Processed dimensions:", cleaned.shape)
print("Figures:", len(manifest["figures"]), "Tables:", len(manifest["table_files"]))
print("Business report:", REPORTS / "PHASE2_BUSINESS_INSIGHTS.md")

def show_chart(name, width=1000):
    display(Image(filename=str(REPORTS / "figures" / f"{name}.png"), width=width))

## B. Target analysis and KPIs

Churn proportion = count(Churn=Yes) / all customers. Retention proportion here means the share labeled No in this snapshot, not an operational retention measure over a defined time period. Multiply proportions by 100 to display percentages. Majority-class imbalance makes raw class counts important context.

In [ ]:
kpis = customer_kpis(cleaned)
display(pd.Series(kpis, name="value"))
show_chart("churn_distribution")
display(Markdown(f"**Observed:** {kpis['churn_count']:,}/{kpis['total_customers']:,} customers "
    f"are labeled churned ({kpis['churn_proportion']:.2%}); {kpis['retained_count']:,} are labeled No "
    f"({kpis['retention_proportion']:.2%}). This sample is imbalanced; these are not calendar-month rates."))

## C. Univariate analysis

Univariate analysis studies one variable at a time. Numerical summaries include the sample standard deviation, quartiles and IQR = Q3 − Q1. IQR fences flag potential outliers for review; they do not prove an error. Histograms show distribution shape. The boxplots later compare these summaries across outcome groups.

Categorical summaries show counts and proportions, preserving structural service categories.

In [ ]:
display(numerical_summary(cleaned))
show_chart("numerical_histograms", width=1200)
display(categorical_summary(cleaned))
show_chart("categorical_frequency_counts", width=1400)

## D. Bivariate analysis — churn and customer characteristics

Bivariate analysis examines two variables. For each category, the denominator is the number of customers in that category. `groupby(...)["ChurnFlag"].mean()` is a segment proportion because the flag is 0/1. Count plus mean is more informative than churn counts alone.

Rate charts use 0–100% axes, show denominators and 95% Wilson intervals. Intervals quantify binomial-model uncertainty; they are not evidence of a random representative sample, and interval overlap is not used as a significance test.

In [ ]:
for feature, chart in [("Contract", "churn_rate_by_contract"),
                       ("TenureGroup", "churn_rate_by_tenure_group"),
                       ("InternetService", "churn_rate_by_internet_service"),
                       ("PaymentMethod", "churn_rate_by_payment_method")]:
    display(churn_by_segment(cleaned, feature))
    show_chart(chart)
contracts = churn_by_segment(cleaned, "Contract").set_index("Contract")
display(Markdown(f"**Observed:** month-to-month churn is "
    f"{contracts.loc['Month-to-month', 'churn_rate']:.2%} "
    f"(n={int(contracts.loc['Month-to-month', 'customer_count']):,}) versus "
    f"{contracts.loc['Two year', 'churn_rate']:.2%} for two-year contracts "
    f"(n={int(contracts.loc['Two year', 'customer_count']):,}). "
    "This motivates investigation of contract mix and customer choices; it does not demonstrate an effect of changing contracts."))

In [ ]:
display(numerical_summary(cleaned, by_churn=True))
show_chart("tenure_distribution_by_churn")
show_chart("monthly_charges_distribution_by_churn")
show_chart("numerical_boxplots_by_churn", width=1200)
means = cleaned.groupby("Churn")["MonthlyCharges"].mean()
display(Markdown(f"**Observed:** mean monthly charges are {means['Yes']:.2f} for churned "
    f"and {means['No']:.2f} for non-churned customers. Plan mix and tenure may confound this comparison; no causal price effect is estimated."))

### Selected services and senior-citizen status

Review technical support, online security and SeniorCitizen as categorical attributes. Distinguish subscribers without an add-on from customers without internet service. A support subscription does not measure support quality or contact history.

In [ ]:
for feature in ["TechSupport", "OnlineSecurity", "SeniorCitizen"]:
    display(churn_by_segment(cleaned, feature))
show_chart("churn_by_selected_services", width=1400)
internet_subscribers = cleaned.loc[cleaned.InternetService.ne("No")]
display(churn_by_segment(internet_subscribers, "OnlineSecurity"))

### Tenure, monthly charges and cumulative charges

Scatterplots examine both requested charge relationships using all complete pairs. Pearson correlation summarizes linear association; Spearman summarizes monotonic rank association. Only tenure and the two charge measures enter these matrices. Do not correlate arbitrary category codes. Missing TotalCharges reduces pairwise coverage; IDs and target-derived features are excluded.

In [ ]:
correlations = correlation_tables(cleaned)
for name, table in correlations.items():
    print(name)
    display(table)
show_chart("numerical_correlation_heatmap")
show_chart("tenure_vs_total_charges")
show_chart("monthly_charges_vs_total_charges")

## E. Selected multivariate analysis

Multivariate analysis examines combinations of three or more variables: two grouping attributes plus churn. Fixed monthly-charge groups are <50, 50 to <80, and 80+ dataset charge units. Every heatmap cell includes its size. Empty combinations are blank, not zero-churn populations; groups below 30 are flagged. Stratifying by a few attributes does not remove all confounding.

In [ ]:
for features, chart in [(["Contract", "TenureGroup"], "churn_contract_tenure"),
                        (["InternetService", "TechSupport"], "churn_internet_technical_support"),
                        (["Contract", "MonthlyChargeGroup"], "churn_contract_monthly_charges")]:
    display(churn_by_segment(cleaned, features))
    show_chart(chart)
display(Markdown("**Interpretation:** investigate whether contract, tenure and service mix help explain aggregate differences. These descriptive strata do not establish what would happen if a customer's service or contract changed."))

## Exploratory statistical support

A fixed family of eight marginal Pearson chi-square tests checks categorical independence. Holm adjustment addresses multiple testing within this family. The output includes Cramér's V, degrees of freedom and minimum observed/expected counts. Support is flagged only when adjusted p<0.05 and all observed/expected cells are at least five. Small or degenerate groups cannot silently become significant results.

These are exploratory associations, not causal or confirmatory conclusions. Wilson intervals are marginal, not simultaneous. References: [SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html), [NIST](https://www.itl.nist.gov/div898/handbook/prc/section2/prc241.htm), [Holm correction](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html).

In [ ]:
tests = association_tests(cleaned)
display(tests)
display(Markdown(f"**Observed:** {int(tests.supports_association.sum())} of {len(tests)} "
    "exploratory feature-level tests meet the stated criteria. Check effect sizes and segment counts; small p-values alone do not establish practical impact."))

## Business interpretation and limitations

Read the [generated business insights report](../reports/PHASE2_BUSINESS_INSIGHTS.md). For each major finding it includes evidence, business relevance, a hypothesis to test and the additional evidence required. Ideas include researching onboarding, contract choices, billing clarity and service experience. None is a proven retention action.

No dates support monthly trends, no counterfactual outcomes support causal claims, and no costs support retention ROI. Preserve charge coverage and sample-size caveats. Later modeling must separate training/evaluation data before learned preprocessing and keep customerID, Churn and ChurnFlag out of predictive features. Full-dataset EDA means future evaluation data has already been inspected descriptively; disclose that limitation and seek unseen data for confirmation.

See [the Phase 2 guide](../docs/PHASE2_GUIDE.md) for file explanations and interview questions. Stop after this phase.

In [ ]:
assert hashlib.sha256(DEFAULT_RAW_PATH.read_bytes()).hexdigest() == raw_sha
assert not validate_cleaned_dataset(cleaned)["errors"]
assert kpis["churn_count"] + kpis["retained_count"] == len(cleaned)
assert all(Path(p).is_file() for p in manifest["figures"])
assert all(Path(p).is_file() for p in manifest["table_files"])
print("Phase 2 notebook checks passed. Source fingerprint unchanged:", raw_sha)